# Shikibu 学習（Colab 用・汎用）

このノートブックは、Google Colab の GPU ランタイムで評価関数の学習を回すためのものです。
学習データ・出力先・リポジトリの場所は、利用者が決めます。**節 2 の設定セルだけを書き換えれば使えます。**

- 学習は裏のプロセスで走るので、表示のセルを止めても、ブラウザを再読み込みしても続きます。
- 途中で切れても、同じ手順で続きから再開できます（`docs/OPERATIONS.md` の 2）。
- 学習が終わったら、重みファイル（`.shk`）を書き出し、量子化の確認をします。
- 同じ出力フォルダで2つの学習が同時に動くことはありません（ロック `train.lock`）。

## 1. Google Drive をマウントする

Drive にあるリポジトリ・wheel・データ・出力を使うため、最初にマウントします。

In [ ]:
# Google Drive を /content/drive にマウントする（Colab でだけ使える）
from google.colab import drive
drive.mount('/content/drive')

## 2. 設定（ここだけ書き換える）

`<…>` の目印を、自分の環境の値に書き換えてください。目印が残っていると、次のセルでエラーになります。

- `REPO_URL`・`BRANCH`：リポジトリの取得元。Drive に `REPO` が無いときだけ使う。
- `REPO`：Drive 上のリポジトリの場所。
- `WHEEL_DIR`：`shk_feat` の wheel の保存先（無ければ初回にビルドして保存する）。
- `DATA_GLOB`：学習用の PSV のパス（`*.psv` のように glob で書く）。
- `OUT_ROOT`・`RUN_NAME`：出力は `OUT_ROOT/RUN_NAME` に作られる。
- `TEMPLATE`：ひな形の設定（`configs/colab_v0.toml` は RayPair_v0、`configs/colab_v1.toml` は RayPair_v1）。
- `CACHE_DIR`：Colab のローカルの一時コピー先（Drive には置かない）。
- `SESSION_MAX_STEPS`：このセッションで区切る通算の更新回数。`None` なら設定の `max_steps`。

In [ ]:
# ===== 設定：<…> の目印を、自分の環境の値に書き換える =====
REPO_URL = "https://github.com/<GitHub のユーザー名>/Shikibu.git"   # リポジトリの取得元（Drive に無いときだけ使う）
BRANCH = "main"                                                    # 取得するブランチ
REPO = "/content/drive/MyDrive/<リポジトリを置くフォルダ>/Shikibu"     # Drive 上のリポジトリの場所
WHEEL_DIR = "/content/drive/MyDrive/<wheel を置くフォルダ>"             # shk_feat の wheel の保存先
DATA_GLOB = "/content/drive/MyDrive/<学習データのフォルダ>/*.psv"        # 学習用の PSV
OUT_ROOT = "/content/drive/MyDrive/<出力のフォルダ>"                     # 学習の出力の親フォルダ
RUN_NAME = "<学習の名前>"                                               # 出力フォルダ名（OUT_ROOT/RUN_NAME）
TEMPLATE = "configs/colab_v0.toml"                                   # ひな形の設定（RayPair_v1 は configs/colab_v1.toml）
CACHE_DIR = "/content/shk_cache"                                     # ローカルの一時コピー先（Colab の /content）
SESSION_MAX_STEPS = None                                             # None なら設定の max_steps。数値なら通算の更新回数の上限

# ===== 目印の確認（ここで止まったら、上の <…> を書き換える） =====
_settings = {"REPO_URL": REPO_URL, "REPO": REPO, "WHEEL_DIR": WHEEL_DIR, "DATA_GLOB": DATA_GLOB,
             "OUT_ROOT": OUT_ROOT, "RUN_NAME": RUN_NAME}
_left = [k for k, v in _settings.items() if "<" in v and ">" in v]
if _left:
    raise ValueError("設定セルの <…> を書き換えてください: " + ", ".join(_left))
print("設定を確認しました。")

## 3. リポジトリを取得（または更新）する

`REPO` があれば `git pull` で更新し、無ければ `REPO_URL` から `git clone` します。

In [ ]:
import os
import subprocess
import glob

os.makedirs(WHEEL_DIR, exist_ok=True)
if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
print("リポジトリ:", REPO, "/ ブランチ:", BRANCH)

## 4. wheel（`shikibu_feat`）をインストールする

- `WHEEL_DIR` に `shk_feat-*.whl` があれば、それを入れます（数秒）。
- 無ければ Rust を入れて `maturin build --release` で wheel を作り、`WHEEL_DIR` に保存してから入れます（初回だけ）。
- Rust のソースを変えたときは、`WHEEL_DIR` の wheel を消してから実行し直してください。

In [ ]:
# 既存の wheel を探す。複数あれば一番新しいものを使う
wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)

if wheels:
    subprocess.run(["pip", "install", "-q", "--force-reinstall", wheels[-1]], check=True)
    print("保存済みの wheel を入れました:", wheels[-1])
else:
    if subprocess.run(["bash", "-lc", "command -v cargo"], capture_output=True).returncode != 0:
        subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
    cargo_env = os.path.expanduser("~/.cargo/bin")
    env = dict(os.environ, PATH=cargo_env + os.pathsep + os.environ["PATH"])
    subprocess.run(["pip", "install", "-q", "maturin"], check=True)
    subprocess.run(["maturin", "build", "--release", "-m", "crates/shk_feat/Cargo.toml",
                    "-o", WHEEL_DIR], cwd=REPO, env=env, check=True)
    built = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)
    subprocess.run(["pip", "install", "-q", "--force-reinstall", built[-1]], check=True)
    print("wheel をビルドして保存しました:", built[-1])

import torch
import shikibu_feat
print("torch", torch.__version__, "/ CUDA:", torch.cuda.is_available(), "/ GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし")
print("論理コア数:", os.cpu_count(), "/ shikibu_feat OK, v0 の次元数:", shikibu_feat.num_features(0))

## 5. 設定ファイルを出力フォルダに書き出す

ひな形（`TEMPLATE`）の `<…>` の値を、上の設定セルの値で置き換えて、`OUT_ROOT/RUN_NAME/config.toml` に書き出します。
学習はこの書き出したファイルを使います。Drive に残るので、再開のときもそのまま使えます。
リポジトリのひな形は変えません。

In [ ]:
import re
import tomllib

def set_key(text, key, value):
    """ひな形の `key = …` の行を、value の値に置き換える（行はちょうど1つでなければならない）。"""
    if isinstance(value, bool):
        lit = "true" if value else "false"
    elif isinstance(value, str):
        lit = '"%s"' % value
    else:
        lit = repr(value)
    new, n = re.subn(rf"^{key} = .*$", f"{key} = {lit}", text, flags=re.M)
    if n != 1:
        raise ValueError(f"ひな形に {key} の行が1つだけ無い（{n} 個）")
    return new

OUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)
text = open(os.path.join(REPO, TEMPLATE), encoding="utf-8").read()
text = set_key(text, "train_glob", DATA_GLOB)
text = set_key(text, "cache_dir", CACHE_DIR)
text = set_key(text, "out_dir", OUT_DIR)
if SESSION_MAX_STEPS is not None:
    text = set_key(text, "max_steps", SESSION_MAX_STEPS)

os.makedirs(OUT_DIR, exist_ok=True)
CONFIG = os.path.join(OUT_DIR, "config.toml")
with open(CONFIG, "w", encoding="utf-8") as f:
    f.write(text)
cfg = tomllib.loads(text)

shards = sorted(glob.glob(DATA_GLOB))
print("学習用の PSV:", len(shards), "個")
for p in shards:
    print("  ", os.path.basename(p), "%.2f GB" % (os.path.getsize(p) / 1e9))
print("設定:", CONFIG)
print("出力先:", OUT_DIR, "/ checkpoint:", "あり（--resume で続きから）" if os.path.exists(os.path.join(OUT_DIR, "checkpoint.pt")) else "なし（最初から）")
print("バッチ:", cfg["train"]["batch_size"], "/ 総更新回数:", cfg["train"]["max_steps"], "/ シャッフルバッファ:", cfg["data"]["shuffle_buffer"])

## 6. 学習を裏で開始する（`mon.start`）

- `checkpoint.pt` があれば `--resume` を付けて続きから学習します。無ければ最初から学習します。
- 同じ出力フォルダで学習が実行中なら、学習器はロックで断り、次の節の表示に理由が出ます。
- 切断の直後で、ロックが残ったまま学習が動いていないとき（`docs/OPERATIONS.md` の 3）だけ、`force=True` にします。
- 止めるときは `mon.stop()` を実行します（最後の保存より後の分は失われます）。

In [ ]:
import sys

sys.path.insert(0, os.path.join(REPO, "tools"))
import shk_monitor as mon

FORCE_UNLOCK = False   # 切断の直後で、学習が動いていないのにロックが残っているときだけ True にする
mon.start(CONFIG, repo=REPO, force=FORCE_UNLOCK)

## 7. 進み具合を表示する（`mon.watch`）

15秒ごとに、段階・進捗・残り時間・速さ・学習曲線を表示します。学習の間は、このセルを動かし続けてください。

In [ ]:
mon.watch(CONFIG)

## 8. `.shk`（重みファイル）を書き出す

学習の途中の `checkpoint.pt` からでも書き出せます。書き出したあと、読み直して中身を確かめます。

In [ ]:
import subprocess

SHK = os.path.join(OUT_DIR, RUN_NAME + ".shk")
env = dict(os.environ, PYTHONPATH=os.path.join(REPO, "python"))
r = subprocess.run(["python", "-m", "shikibu.export",
                    "--checkpoint", os.path.join(OUT_DIR, "checkpoint.pt"), "--out", SHK],
                   cwd=REPO, env=env)
print("終了コード:", r.returncode)

## 9. 量子化の確認（`quantsim`）

エンジンで使う整数の量子化を Python で再現し、float32 の重みとの差を測ります。
表示される「z の差」「評価値の差（cp）」「損失」を控えてください。

注意：ここで使う PSV は学習に使ったファイルの先頭です。本当の確認には、学習に使わなかった局面（検証用）を使ってください。

In [ ]:
first_psv = sorted(glob.glob(DATA_GLOB))[0]
r = subprocess.run(["python", "-m", "shikibu.quantsim", "--shk", SHK, "--psv", first_psv, "--n", "100000"],
                   cwd=REPO, env=env)
print("終了コード:", r.returncode)